# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, how deep (in z, µm) real tissue signal actually extends, using one already-finished round (default: `cells`).

Procedure:
1. Resolve the target round's frame table; pick the frame index for `CHANNEL_NM` closest to `Z_REFERENCE_UM`.
2. For every FOV, get just its **reference frame**'s histogram (one frame) -- reused from this notebook's own cache or an existing full per-frame histogram where possible, computed fresh only otherwise (`analysis.fov.get_histogram_for_frames`). Cheap regardless of the round's total frame count.
3. Pool every FOV's reference-frame histogram and estimate a binarization threshold as the valley between its two most prominent peaks (reuses `acquisition.mosaic._estimate_bimodal_threshold`'s exact peak-finding algorithm, adapted to the linear-binned histograms this pipeline already saves rather than the log-rebinned raw pixels `02_create_boundary_from_mosaic.ipynb` uses -- review the plot and override `THRESHOLD` manually if the auto-estimate looks wrong).
4. For every FOV, scan z-planes of `CHANNEL_NM` shallow -> deep counting true-pixels (NTP, intensity >= `THRESHOLD`) directly, **stopping as soon as a z-plane's NTP first drops to/below `NTP_THRESHOLD`** -- deeper frames are then never read at all (`analysis.fov.measure_tissue_ntp_profile`; sourced from an existing full histogram with no stack read at all when one's already on disk, via `analysis.fov.ntp_profile_from_histogram`). This assumes tissue signal only decreases with depth past its edge.
5. Lay every FOV's last-passing z out on its stage-position grid and plot as a heatmap.

Figures and results are saved under `SAMPLE_DIR/analysis/figures/` and `SAMPLE_DIR/analysis/` respectively, in addition to being shown inline. `z=1` in the `(z, channel)=(1, 405)` example is interpreted as a **literal micron stage position** (nearest available z-step for that channel is used), not an ordinal step index.

Runs anywhere the standard `SAMPLE_DIR/{data,metadata,positions,analysis}` layout is reachable, including a cluster node (same convention as `05_batch_sample_review.ipynb`/`07_cluster_submit_analysis.ipynb`). Steps 2 and 4's backfill loops are intentionally sequential, not process-pool-parallelized: on a shared SLURM node, `os.cpu_count()` reports the node's total core count, not this job's actual memory allocation, so sizing a worker pool off it (`config.resolved_n_workers`) can spawn far more workers than the job's real memory allows -- each holding a stack in memory at once -- and get OOM-killed (`BrokenProcessPool`). Reading only what's actually needed (one reference frame per FOV in step 2; only the z-planes up to the first failing one in step 4) keeps the sequential version fast without a pool.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter
from MERci.analysis.fov       import (
    load_histogram, get_histogram_for_frames,
    measure_tissue_ntp_profile, ntp_profile_from_histogram,
    save_ntp_profile, load_ntp_profile,
)
from MERci.acquisition.configs import find_frame_table_for_hal_config
from MERci.acquisition.mosaic  import _estimate_bimodal_threshold

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Reference channel + z (micron stage position, NOT an ordinal step index -- the nearest
# available z-step for CHANNEL_NM is used) for pooling the threshold-finding histogram.
CHANNEL_NM     = 405.0
Z_REFERENCE_UM = 1.0

# A z-plane still counts as "has tissue" if its true-pixel count (NTP) exceeds this.
NTP_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Binarization intensity threshold; None = auto-estimate from the pooled reference-frame
# histogram (section 5) -- review that plot before trusting the estimate on a new experiment.
THRESHOLD = None

print(f"Sample name         : {SAMPLE_NAME}")
print(f"Round imaging type  : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel / Z reference: {CHANNEL_NM} nm / {Z_REFERENCE_UM} um")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# This notebook's own cache -- deliberately NOT tracker.histogram_path()'s canonical
# location: both caches below are partial (one frame, or an early-stopped z-range),
# so writing them to the path routine analysis (analyze_file/compute_histogram_only)
# uses for a FULL histogram would make analyze_file think that FOV is already done
# and skip ever writing the real, complete histogram.
tissue_cache_dir   = config.analysis_dir / "tissue_thickness_cache"
reference_hist_dir = tissue_cache_dir / "reference_histograms"
ntp_profile_dir    = tissue_cache_dir / "ntp_profiles"
reference_hist_dir.mkdir(parents=True, exist_ok=True)
ntp_profile_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {tissue_cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

reference_frame_idx = int((channel_frames["z"] - Z_REFERENCE_UM).abs().idxmin())
reference_z_um       = float(channel_frames.loc[reference_frame_idx, "z"])
z_frame_indices      = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")
print(f"Reference frame: frame_idx={reference_frame_idx}, z={reference_z_um:.2f} um "
      f"(closest to requested {Z_REFERENCE_UM} um)")

## 4 — Load (or backfill) each FOV's reference-frame histogram

Only the single reference frame is needed here, to pool into a global threshold
estimate (section 5) -- cheap regardless of the round's total frame count. Reuse
priority per FOV: this notebook's own cache (`analysis/tissue_thickness_cache/
reference_histograms/`) first; then, if routine QC analysis
(`01_fov_scheduler.ipynb`/`07_cluster_submit_analysis.ipynb`) has already written a
**full** per-frame histogram for this FOV, its reference-frame row is extracted
directly -- no disk read of the image stack at all; only as a last resort is a
fresh single-frame histogram computed (`analysis.fov.get_histogram_for_frames`) and
cached into this notebook's own folder (never into the canonical histogram path
routine analysis uses -- see the note in section 2's setup cell).

In [ ]:
def reference_hist_path(fpath):
    return reference_hist_dir / f"{Path(fpath).stem}_ref_hist.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

fov_reference_hist = {}   # fov_id -> {"counts": <1-row array>, "bin_centers", "bin_edges"}
from_own_cache, from_full_histogram, to_compute = [], [], []
n_missing_on_disk = 0

for fpath in files:
    if reference_hist_path(fpath).exists():
        from_own_cache.append(fpath)
    elif tracker.histogram_path(fpath).exists():
        from_full_histogram.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    fov_reference_hist[meta.fov_id_of_file(fpath)] = load_histogram(reference_hist_path(fpath))

for fpath in from_full_histogram:
    full_hist = load_histogram(tracker.histogram_path(fpath))
    fov_reference_hist[meta.fov_id_of_file(fpath)] = {
        "counts":      full_hist["counts"][reference_frame_idx][np.newaxis, :],
        "bin_centers": full_hist["bin_centers"],
        "bin_edges":   full_hist["bin_edges"],
    }

print(f"{len(from_own_cache)} reference histogram(s) from this notebook's own cache, "
      f"{len(from_full_histogram)} extracted from an existing full per-frame histogram "
      f"(no stack read needed).")

n_computed = 0
if to_compute:
    print(f"Computing {len(to_compute)} missing reference-frame histogram(s) sequentially "
          f"(1 frame/FOV, channel {CHANNEL_NM:.0f} nm at z={reference_z_um:.1f} um).")
    reporter = ProgressReporter(total=len(to_compute), label="Computing reference-frame histograms")
    for fpath in reporter.wrap(to_compute):
        hist = get_histogram_for_frames(
            fpath, reference_hist_path(fpath), [reference_frame_idx],
            frame_width=config.frame_width, frame_height=config.frame_height,
            histogram_bins=config.histogram_bins, histogram_range=config.histogram_range,
        )
        fov_reference_hist[meta.fov_id_of_file(fpath)] = hist
        n_computed += 1

print(f"Reference-frame histograms ready for {len(fov_reference_hist)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Pool the reference-frame histogram and estimate a threshold

Review the plot -- if the estimate looks wrong (or no threshold is found at all, e.g. a
genuinely unimodal distribution), set `THRESHOLD` by hand in the cell above and re-run
from here.

In [ ]:
bin_centers   = next(iter(fov_reference_hist.values()))["bin_centers"]
bin_edges     = next(iter(fov_reference_hist.values()))["bin_edges"]
pooled_counts = np.zeros_like(bin_centers)

fig, ax = plt.subplots(figsize=(8, 5))
for hist in fov_reference_hist.values():
    fov_counts = hist["counts"][0]
    pooled_counts += fov_counts
    ax.plot(bin_centers, fov_counts, "-", color=(0.7, 0.7, 0.7), alpha=0.25, lw=0.6)
ax.plot(bin_centers, pooled_counts, "-", color="black", lw=1.8, label="all FOVs combined")

# Reuses mosaic.py's peak-finding valley estimator verbatim: it expects LOG-space bin
# centers and returns 10**valley (linear units). Our histogram bins are already linear
# (fixed at acquisition time -- see ExperimentConfig.histogram_range), so passing
# log10(bin_centers) round-trips back to the correct linear threshold without
# duplicating the peak-finding logic. Unlike mosaic.py's own use (log-rebinned raw
# pixels, chosen specifically to spread out the bimodal peaks for this kind of
# estimate), our bins are fixed-linear -- same algorithm, but peak separation may look
# different since the bins weren't chosen with this in mind.
log_bin_centers      = np.log10(np.clip(bin_centers, 1, None))
estimated_threshold  = _estimate_bimodal_threshold(log_bin_centers, pooled_counts)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm, z={reference_z_um:.1f} um)")
ax.set_ylabel("Pixel count (summed across FOVs)")
title = f"Round {target_round_id} -- reference-frame histogram ({len(fov_reference_hist)} FOVs)"
if estimated_threshold is not None:
    ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
               label=f"estimated threshold = {estimated_threshold:.0f}")
    title += f"\nestimated threshold: {estimated_threshold:.0f}"
ax.set_title(title)
ax.legend()
fig.tight_layout()

fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
    if THRESHOLD is None:
        raise ValueError(
            "Could not auto-estimate a threshold (not clearly bimodal) -- set THRESHOLD manually above."
        )
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: sequential true-pixel-count (NTP) scan, stopping at the first failing z

For each FOV, scans z-planes shallow -> deep, counting pixels with intensity >=
`THRESHOLD` directly (no histogram needed, since `THRESHOLD` is already fixed by
section 5) and **stops as soon as a z-plane's count first drops to/below
`NTP_THRESHOLD`** -- every deeper frame is then assumed to also be past the tissue,
so it's never read. This assumes tissue signal only decreases with depth past its
edge: a real signal that dips below threshold and reappears deeper would be missed
(the previous approach -- read every z, report the deepest one that ever passes --
was robust to that, at the cost of reading/histogramming every frame regardless).

Reuse priority per FOV, same as section 4: this notebook's own cache
(`analysis/tissue_thickness_cache/ntp_profiles/`) first; then, if a full per-frame
histogram already exists, the NTP profile is computed from its bins with no stack
read at all; only as a last resort does this run the actual sequential raw-pixel
scan.

In [ ]:
def ntp_profile_path(fpath):
    return ntp_profile_dir / f"{Path(fpath).stem}_ntp_profile.npz"


fov_of_file = {fpath: meta.fov_id_of_file(fpath) for fpath in files}

from_own_cache, from_full_histogram, to_scan = [], [], []
n_missing_on_disk = 0
for fpath in files:
    if ntp_profile_path(fpath).exists():
        from_own_cache.append(fpath)
    elif tracker.histogram_path(fpath).exists():
        from_full_histogram.append(fpath)
    elif fpath.exists():
        to_scan.append(fpath)
    else:
        n_missing_on_disk += 1

profiles = {}
for fpath in from_own_cache:
    profiles[fov_of_file[fpath]] = load_ntp_profile(ntp_profile_path(fpath))

for fpath in from_full_histogram:
    full_hist = load_histogram(tracker.histogram_path(fpath))
    profile   = ntp_profile_from_histogram(full_hist, z_frame_indices, THRESHOLD, NTP_THRESHOLD)
    profiles[fov_of_file[fpath]] = profile
    save_ntp_profile(ntp_profile_path(fpath), profile)   # so a later run skips even this

print(f"{len(from_own_cache)} NTP profile(s) from this notebook's own cache, "
      f"{len(from_full_histogram)} computed in-memory from an existing full histogram "
      f"(no stack read needed).")

n_scanned = 0
if to_scan:
    print(f"Sequentially scanning {len(to_scan)} FOV(s) with no cached histogram -- "
          f"stopping each as soon as a z-plane's true-pixel count first drops to/below "
          f"NTP_THRESHOLD.")
    reporter = ProgressReporter(total=len(to_scan), label="Scanning FOV z-stacks")
    for fpath in reporter.wrap(to_scan):
        profile = measure_tissue_ntp_profile(
            fpath, z_frame_indices, THRESHOLD, NTP_THRESHOLD,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        profiles[fov_of_file[fpath]] = profile
        save_ntp_profile(ntp_profile_path(fpath), profile)
        n_scanned += 1

print(f"NTP profiles ready for {len(profiles)} / {len(files)} FOVs "
      f"({n_scanned} newly scanned this run, {n_missing_on_disk} not yet written on disk).")

results = []
for fov_id, profile in profiles.items():
    results.append({
        "fov_id":        fov_id,
        "last_z_um":     profile["last_z_um"],
        "stopped_early": profile["stopped_early"],
        "x_um":          meta.fovs[fov_id].position[0],
        "y_um":          meta.fovs[fov_id].position[1],
    })

results_df = pd.DataFrame(results)
n_no_signal = results_df["last_z_um"].isna().sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above NTP_THRESHOLD at all.")
print(results_df["last_z_um"].describe())

## 7 — Tissue-thickness heatmap across the FOV grid

In [ ]:
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    04_view_intensity_stats.ipynb's heatmap: round to the nearest integer micron, then
    rank each axis's unique values -- robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1

matrix = np.full((n_y, n_x), np.nan)
for _, row in results_df.iterrows():
    xi, yi = grid[row["fov_id"]]
    if pd.notna(row["last_z_um"]):
        matrix[yi, xi] = row["last_z_um"]

fig, ax = plt.subplots(figsize=(max(5, n_x * 0.4 + 1.5), max(4, n_y * 0.4 + 1.5)))
im = ax.imshow(matrix, cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.04)
cbar.set_label("Last z with tissue signal (um)")
ax.set_title(f"Round {target_round_id} -- tissue thickness map ({len(fov_ids)} FOVs)")
ax.set_xlabel("X grid index  (increasing stage X \u2192)")
ax.set_ylabel("Y grid index  (increasing stage Y \u2193)")
fig.tight_layout()

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")